# 02. Descriptiva detallada y extracción de características

Este cuaderno transforma la información original en variables útiles para modelar, y revisa la relación entre estas nuevas variables y la ocurrencia de retrasos.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use('seaborn-v0_8-darkgrid')
df = pd.read_csv(Path('data/public_transport_delays.csv'))

df['event_type'] = df['event_type'].fillna('Sin evento')
df['event_presence'] = (df['event_type'] != 'Sin evento').astype(int)
df['date'] = pd.to_datetime(df['date'])
df['time'] = pd.to_datetime(df['time'], format='%H:%M:%S')
df['hour'] = df['time'].dt.hour
df['day_of_week'] = df['date'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

df['scheduled_departure_dt'] = pd.to_datetime(df['date'].astype(str) + ' ' + df['scheduled_departure'])
df['scheduled_arrival_dt'] = pd.to_datetime(df['date'].astype(str) + ' ' + df['scheduled_arrival'])
df['scheduled_trip_duration_min'] = (df['scheduled_arrival_dt'] - df['scheduled_departure_dt']).dt.total_seconds() / 60
df['delay_gap_min'] = df['actual_arrival_delay_min'] - df['actual_departure_delay_min']

df[['hour', 'day_of_week', 'is_weekend', 'event_presence', 'scheduled_trip_duration_min', 'delay_gap_min']].head()

## Estadísticas descriptivas por categoría

Se resumen las principales métricas para detectar diferencias entre grupos y confirmar qué variables aportan contexto al retraso.

In [ ]:
summary = df.groupby('transport_type').agg(
    retrasos=('delayed', 'mean'),
    viajes=('trip_id', 'count'),
    temperatura_prom=('temperature_C', 'mean'),
    congestion_prom=('traffic_congestion_index', 'mean')
).reset_index()
summary

## Relación entre variables y retrasos

Se calculan medias y proporciones para cuantificar cómo cambian los retrasos según condiciones externas y del servicio.

In [ ]:
grouped_weather = df.groupby('weather_condition')['delayed'].mean().sort_values(ascending=False)
grouped_event = df.groupby('event_presence')['delayed'].mean()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
grouped_weather.plot(kind='bar', ax=axes[0], color='#4C78A8')
axes[0].set_title('Probabilidad de retraso por clima')
axes[0].set_ylabel('Proporción')

grouped_event.plot(kind='bar', ax=axes[1], color=['#F58518', '#4C78A8'])
axes[1].set_title('Probabilidad de retraso con/sin evento')
axes[1].set_ylabel('Proporción')
fig.tight_layout()
plt.show()

## Matriz de correlación

Para identificar variables de interés, se construye una matriz con variables numéricas y con representaciones binarias de variables categóricas.

In [ ]:
feature_frame = df[['delayed', 'actual_departure_delay_min', 'actual_arrival_delay_min', 'temperature_C', 'humidity_percent', 'wind_speed_kmh', 'precipitation_mm', 'event_attendance_est', 'traffic_congestion_index', 'holiday', 'peak_hour', 'weekday', 'hour', 'day_of_week', 'is_weekend', 'event_presence', 'scheduled_trip_duration_min', 'delay_gap_min']]
feature_frame = pd.get_dummies(feature_frame, columns=['weekday'], prefix='weekday')
correlation = feature_frame.corr(numeric_only=True)
plt.figure(figsize=(14, 10))
sns.heatmap(correlation, cmap='coolwarm', center=0, annot=False)
plt.title('Matriz de correlación de variables de interés')
plt.tight_layout()
plt.show()

## Variables de interés para el modelo

Se identifican las variables más prometedoras para el entrenamiento: condiciones climáticas, congestión, eventos, hora y duración programada del viaje.

In [ ]:
corr_with_target = correlation['delayed'].drop('delayed').sort_values(ascending=False)
corr_with_target.head(10)

### Resumen del notebook 2

La ingeniería de características permitió convertir el contexto temporal y operativo en variables más informativas, y la matriz de correlación evidenció que la congestión, los retrasos anteriores, la presencia de eventos y la hora del día son variables de alto interés.